In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/kimanaru/dataset-tien-xu-li/val_ml_clean_text.csv
/kaggle/input/datasets/kimanaru/dataset-tien-xu-li/train_ml_clean_text.csv
/kaggle/input/datasets/kimanaru/dataset-tien-xu-li/test_ml_clean_text.csv


In [2]:
import pandas as pd
from pathlib import Path

RESULT_DIR = Path("/kaggle/input/datasets/kimanaru/dataset-tien-xu-li")

train_df = pd.read_csv(RESULT_DIR / "train_ml_clean_text.csv")
val_df = pd.read_csv(RESULT_DIR / "val_ml_clean_text.csv")
test_df = pd.read_csv(RESULT_DIR / "test_ml_clean_text.csv")

train_df["ml_clean_text"] = train_df["ml_clean_text"].fillna("").astype(str)
val_df["ml_clean_text"] = val_df["ml_clean_text"].fillna("").astype(str)
test_df["ml_clean_text"] = test_df["ml_clean_text"].fillna("").astype(str)

train_df["category"] = train_df["category"].astype(str)
val_df["category"] = val_df["category"].astype(str)
test_df["category"] = test_df["category"].astype(str)

X_train = train_df["ml_clean_text"]
y_train = train_df["category"]

X_val = val_df["ml_clean_text"]
y_val = val_df["category"]

X_test = test_df["ml_clean_text"]
y_test = test_df["category"]

# Thử các model

In [3]:
import warnings

warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names"
)

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import f1_score, classification_report
import pandas as pd
import time
import joblib
import lightgbm as lgb

BOW_MAX_FEATURES = 50000
BOW_NGRAM_RANGE = (1, 2)

RANDOM_STATE = 42
N_JOBS = -1

n_estimators_list = [50, 100, 150, 200]
learning_rate_list = [0.03, 0.05, 0.1]
num_leaves_list = [15, 31, 63]

In [5]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)
y_val_encoded = label_encoder.transform(y_val)
y_test_encoded = label_encoder.transform(y_test)

In [6]:
results = []
best_score = -1
best_model = None
best_params = None

total_runs = (
    len(n_estimators_list)
    * len(learning_rate_list)
    * len(num_leaves_list)
)

run_id = 1

for n_estimators in n_estimators_list:
    for learning_rate in learning_rate_list:
        for num_leaves in num_leaves_list:

            print("=" * 80)
            print(f"Running {run_id}/{total_runs}")
            print("n_estimators:", n_estimators)
            print("learning_rate:", learning_rate)
            print("num_leaves:", num_leaves)

            model = Pipeline([
                ("bow", CountVectorizer(
                    max_features=BOW_MAX_FEATURES,
                    ngram_range=BOW_NGRAM_RANGE,
                    lowercase=False,
                    dtype=np.float32
                )),
                ("lgbm", lgb.LGBMClassifier(
                    boosting_type="gbdt",
                    n_estimators=n_estimators,
                    learning_rate=learning_rate,
                    num_leaves=num_leaves,
                    random_state=RANDOM_STATE,
                    n_jobs=N_JOBS,
                    verbose=-1
                ))
            ])

            start = time.time()

            model.fit(X_train, y_train_encoded)

            y_val_pred = model.predict(X_val)
            val_macro_f1 = f1_score(y_val_encoded, y_val_pred, average="macro")

            elapsed = time.time() - start

            print("Validation Macro F1:", val_macro_f1)
            print("Time:", round(elapsed, 2), "seconds")

            current_params = {
                "n_estimators": n_estimators,
                "learning_rate": learning_rate,
                "num_leaves": num_leaves
            }

            results.append({
                "feature_type": "BoW",
                "max_features": BOW_MAX_FEATURES,
                "ngram_range": BOW_NGRAM_RANGE,
                **current_params,
                "random_state": RANDOM_STATE,
                "val_macro_f1": val_macro_f1,
                "time_seconds": elapsed
            })

            if val_macro_f1 > best_score:
                best_score = val_macro_f1
                best_model = model
                best_params = current_params
                print("New best LightGBM!")

            run_id += 1

results_df = pd.DataFrame(results).sort_values("val_macro_f1", ascending=False)

print("=" * 80)
print("BEST LIGHTGBM PARAMS:", best_params)
print("BEST VALIDATION MACRO F1:", best_score)

results_df

Running 1/36
n_estimators: 50
learning_rate: 0.03
num_leaves: 15
Validation Macro F1: 0.514862120501785
Time: 39.67 seconds
New best LightGBM!
Running 2/36
n_estimators: 50
learning_rate: 0.03
num_leaves: 31
Validation Macro F1: 0.5526960770565729
Time: 50.23 seconds
New best LightGBM!
Running 3/36
n_estimators: 50
learning_rate: 0.03
num_leaves: 63
Validation Macro F1: 0.5800962171906376
Time: 69.68 seconds
New best LightGBM!
Running 4/36
n_estimators: 50
learning_rate: 0.05
num_leaves: 15
Validation Macro F1: 0.5543679125460178
Time: 38.85 seconds
Running 5/36
n_estimators: 50
learning_rate: 0.05
num_leaves: 31
Validation Macro F1: 0.5830126770922455
Time: 50.19 seconds
New best LightGBM!
Running 6/36
n_estimators: 50
learning_rate: 0.05
num_leaves: 63
Validation Macro F1: 0.60615209548537
Time: 69.95 seconds
New best LightGBM!
Running 7/36
n_estimators: 50
learning_rate: 0.1
num_leaves: 15
Validation Macro F1: 0.5908633819893302
Time: 39.3 seconds
Running 8/36
n_estimators: 50
learn

,feature_type,max_features,ngram_range,n_estimators,learning_rate,num_leaves,random_state,val_macro_f1,time_seconds
35,BoW,50000,"(1, 2)",200,0.10,63,42,0.670066,236.690019
26,BoW,50000,"(1, 2)",150,0.10,63,42,0.669393,154.189770
17,BoW,50000,"(1, 2)",100,0.10,63,42,0.660866,116.079752
32,BoW,50000,"(1, 2)",200,0.05,63,42,0.659971,202.748342
34,BoW,50000,"(1, 2)",200,0.10,31,42,0.658464,120.769644
25,BoW,50000,"(1, 2)",150,0.10,31,42,0.654166,106.618277
23,BoW,50000,"(1, 2)",150,0.05,63,42,0.653725,184.499526
29,BoW,50000,"(1, 2)",200,0.03,63,42,0.646954,208.732554
16,BoW,50000,"(1, 2)",100,0.10,31,42,0.643331,74.917593
31,BoW,50000,"(1, 2)",200,0.05,31,42,0.642929,126.960648


In [7]:
y_test_pred = best_model.predict(X_test)

print("Test Macro F1:", f1_score(y_test_encoded, y_test_pred, average="macro"))
print(classification_report(
    y_test_encoded,
    y_test_pred,
    target_names=label_encoder.classes_
))

Test Macro F1: 0.6792381597278613
                precision    recall  f1-score   support

  BLACK VOICES       0.65      0.44      0.53       687
      BUSINESS       0.66      0.58      0.62       899
        COMEDY       0.67      0.47      0.56       810
 ENTERTAINMENT       0.72      0.80      0.76      2605
  FOOD & DRINK       0.77      0.79      0.78       951
HEALTHY LIVING       0.46      0.28      0.35      1004
 HOME & LIVING       0.82      0.71      0.76       648
     PARENTING       0.61      0.72      0.66      1319
       PARENTS       0.49      0.31      0.38       593
      POLITICS       0.85      0.91      0.88      5341
  QUEER VOICES       0.85      0.74      0.79       952
        SPORTS       0.81      0.73      0.77       761
STYLE & BEAUTY       0.84      0.85      0.85      1472
        TRAVEL       0.80      0.80      0.80      1485
      WELLNESS       0.66      0.80      0.73      2692

      accuracy                           0.75     22219
     macro a